# Corrections post-scoring — greenery + exclusions 25mm

À exécuter APRÈS avoir fait tourner Notebook 3, 3bis, 4 et 5.

## Deux corrections
1. **greenery_ratio** recalculé avec catégories strictes (actuellement 73% > 0.8, bug évident)
2. **Exclusions surface** pour vélo de route 25mm (gravel, unpaved, track non asphalté)

## Ce qui se met à jour
- `greenery_ratio` (osm_edges)
- `is_routable` (osm_edges) : exclusions renforcées
- `cost_factor_club/solo` et `score_osm_club/solo` (edge_scores)
- `score_final_club/solo` et `cost_factor_v2_*` (edge_scores)

## Ce qui ne bouge pas
- Score club brut (heatmap)
- Données élévation
- Modèle PU learning
- `score_club_gen`

Temps total : 10-15 min.


## 0. Setup

In [1]:
import pandas as pd
import numpy as np
import time
from sqlalchemy import create_engine, text
import warnings
warnings.filterwarnings("ignore")

DB_CONFIG = {"user": "postgres", "password": "4421",
             "host": "localhost", "port": 5432, "database": "velo_club"}
url = (f"postgresql+psycopg2://{DB_CONFIG['user']}:{DB_CONFIG['password']}"
       f"@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}")
engine = create_engine(url, pool_pre_ping=True)
print("engine OK")

engine OK


## 1. Diagnostic greenery AVANT

Pour comparaison.

In [2]:
with engine.connect() as conn:
    by_hw = pd.read_sql(text("""
        SELECT highway,
               COUNT(*) AS n,
               ROUND(AVG(greenery_ratio)::numeric, 2) AS mean_green,
               ROUND((COUNT(*) FILTER (WHERE greenery_ratio > 0.8) * 100.0 / COUNT(*))::numeric, 1) AS pct_above_80
        FROM osm_edges
        WHERE greenery_ratio IS NOT NULL
          AND highway IN ('primary','secondary','tertiary','residential',
                          'unclassified','cycleway','living_street')
        GROUP BY highway
        ORDER BY mean_green DESC;
    """), conn)
print("Greenery par type (AVANT fix) :")
print(by_hw.to_string(index=False))

Greenery par type (AVANT fix) :
      highway      n  mean_green  pct_above_80
      primary  24221        0.82          80.0
     cycleway  15921        0.82          80.2
 unclassified  26874        0.81          77.5
    secondary  31386        0.81          78.5
     tertiary  33115        0.79          75.4
living_street   5644        0.69          64.5
  residential 136099        0.63          56.9


## 2. Recalcul avec catégories strictes

On garde uniquement les **vraies** zones naturelles :
- `landuse IN ('forest', 'farmland', 'vineyard', 'orchard')`
- `natural IN ('wood', 'scrub', 'heath')`
- `leisure = 'nature_reserve'`

On **supprime** : grass, meadow, recreation_ground, village_green, park, garden (polluent en zone urbaine).

On **ajoute** : filtre taille > 50 000 m² (5 hectares) pour éliminer les petits bouts épars.

In [2]:
print("Extraction zones vertes strictes...")
with engine.begin() as conn:
    conn.execute(text("""
        DROP TABLE IF EXISTS green_zones_strict;
        CREATE TABLE green_zones_strict AS
        SELECT ST_Transform(way, 2154) AS geom,
               ST_Area(ST_Transform(way, 2154)) AS area_m2
        FROM planet_osm_polygon
        WHERE (
            landuse IN ('forest', 'farmland', 'vineyard', 'orchard')
            OR "natural" IN ('wood', 'scrub', 'heath')
            OR leisure = 'nature_reserve'
        )
        AND ST_Area(ST_Transform(way, 2154)) > 50000;
        CREATE INDEX idx_green_strict ON green_zones_strict USING GIST (geom);
        ANALYZE green_zones_strict;
    """))

with engine.connect() as conn:
    row = conn.execute(text("""
        SELECT COUNT(*), ROUND((SUM(area_m2)/1e6)::numeric, 0) FROM green_zones_strict;
    """)).fetchone()
print(f"  {row[0]:,} polygones retenus, {row[1]} km^2 au total")

Extraction zones vertes strictes...
  11,248 polygones retenus, 10785 km^2 au total


In [3]:
import time
from sqlalchemy import text

print("Étape 1 : Création des zones vertes strictes (Extraction + Union + Buffer)...")
t0 = time.time()

with engine.begin() as conn:
    conn.execute(text("""
        DROP TABLE IF EXISTS green_zones_strict_buf;
        
        -- On fait tout d'un coup : on filtre, on transforme, on gonfle de 100m, et on fusionne !
        CREATE TABLE green_zones_strict_buf AS
        SELECT (ST_Dump(ST_Union(ST_Buffer(geom, 100)))).geom AS geom_buf
        FROM (
            SELECT ST_Transform(way, 2154) AS geom
            FROM planet_osm_polygon
            WHERE (
                landuse IN ('forest', 'farmland', 'vineyard', 'orchard')
                OR "natural" IN ('wood', 'scrub', 'heath')
                OR leisure = 'nature_reserve'
            )
            -- Filtre sur la surface (> 5 hectares)
            AND ST_Area(ST_Transform(way, 2154)) > 50000
        ) sub;
        
        -- L'index magique
        CREATE INDEX idx_green_strict_buf ON green_zones_strict_buf USING GIST (geom_buf);
        ANALYZE green_zones_strict_buf;
    """))

print(f"✅ Préparation terminée en {time.time()-t0:.0f}s")

Étape 1 : Création des zones vertes strictes (Extraction + Union + Buffer)...
✅ Préparation terminée en 12s


In [4]:
import time
from sqlalchemy import text

print("Recréation propre de green_zones avec ST_Subdivide...")
t0 = time.time()

with engine.begin() as conn:
    conn.execute(text("""
        DROP TABLE IF EXISTS green_zones_strict_buf;
        
        -- Polygones séparés, buffer 100m, SUBDIVISÉS en petits morceaux ≤256 sommets
        CREATE TABLE green_zones_strict_buf AS
        SELECT ST_Subdivide(ST_Buffer(ST_Transform(way, 2154), 100), 64) AS geom_buf
        FROM planet_osm_polygon
        WHERE (
            landuse IN ('forest', 'farmland', 'vineyard', 'orchard')
            OR "natural" IN ('wood', 'scrub', 'heath')
            OR leisure = 'nature_reserve'
        )
        AND ST_Area(ST_Transform(way, 2154)) > 50000;
        
        CREATE INDEX idx_green_buf ON green_zones_strict_buf USING GIST (geom_buf);
        ANALYZE green_zones_strict_buf;
    """))

with engine.connect() as conn:
    n = conn.execute(text("SELECT COUNT(*) FROM green_zones_strict_buf")).scalar()

print(f"{n:,} petites tuiles de polygones verts en {time.time()-t0:.0f}s")

Recréation propre de green_zones avec ST_Subdivide...
37,227 petites tuiles de polygones verts en 6s


In [5]:
import time
import pandas as pd
from sqlalchemy import text

print("Calcul greenery_ratio par échantillonnage (rapide)...")
t0 = time.time()

with engine.begin() as conn:
    # Reset ultra-rapide via DROP/ADD colonne (instantané, pas de MVCC)
    conn.execute(text("""
        ALTER TABLE osm_edges DROP COLUMN IF EXISTS greenery_ratio;
        ALTER TABLE osm_edges ADD COLUMN greenery_ratio DOUBLE PRECISION DEFAULT 0;
    """))

print(f"Colonne reset en {time.time()-t0:.0f}s")

# Calcul par batch mais avec échantillonnage (ST_Contains sur points)
t1 = time.time()
with engine.connect() as conn:
    min_id, max_id = conn.execute(text("SELECT MIN(edge_id), MAX(edge_id) FROM osm_edges")).fetchone()

BATCH_SIZE = 100_000
n_batches = (max_id - min_id) // BATCH_SIZE + 1

for i in range(n_batches):
    s = min_id + i * BATCH_SIZE
    e = min_id + (i + 1) * BATCH_SIZE
    tb = time.time()
    
    with engine.begin() as conn:
        conn.execute(text("""
            WITH samples AS (
                -- 5 points échantillonnés par arête (0.1, 0.3, 0.5, 0.7, 0.9)
                SELECT e.edge_id,
                       ST_LineInterpolatePoint(e.geom_2154, f.frac) AS pt
                FROM osm_edges e,
                     (VALUES (0.1::float), (0.3), (0.5), (0.7), (0.9)) AS f(frac)
                WHERE e.edge_id >= :s AND e.edge_id < :e
                  AND e.geom_2154 IS NOT NULL
                  AND e.length_m > 0
            ),
            hits AS (
                -- Pour chaque point, est-il dans une zone verte ?
                SELECT s.edge_id, COUNT(DISTINCT g.ctid) > 0 AS hit
                FROM samples s
                LEFT JOIN green_zones_strict_buf g
                  ON ST_Contains(g.geom_buf, s.pt)
                GROUP BY s.edge_id, s.pt
            ),
            ratios AS (
                SELECT edge_id, AVG(hit::int)::float AS ratio
                FROM hits
                GROUP BY edge_id
            )
            UPDATE osm_edges oe
            SET greenery_ratio = r.ratio
            FROM ratios r
            WHERE oe.edge_id = r.edge_id;
        """), {"s": s, "e": e})
    
    print(f"  Batch {i+1}/{n_batches} : {time.time()-tb:.0f}s  total {time.time()-t1:.0f}s")

print(f"\n✅ Calcul terminé en {time.time()-t1:.0f}s total")

# Nettoyage + stats
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS green_zones_strict_buf;"))

with engine.connect() as conn:
    stats = pd.read_sql(text("""
        SELECT
            COUNT(*) FILTER (WHERE greenery_ratio = 0) AS ratio_0,
            COUNT(*) FILTER (WHERE greenery_ratio > 0 AND greenery_ratio <= 0.2) AS r_0_20,
            COUNT(*) FILTER (WHERE greenery_ratio > 0.2 AND greenery_ratio <= 0.5) AS r_20_50,
            COUNT(*) FILTER (WHERE greenery_ratio > 0.5 AND greenery_ratio <= 0.8) AS r_50_80,
            COUNT(*) FILTER (WHERE greenery_ratio > 0.8) AS r_80_100,
            ROUND(AVG(greenery_ratio)::numeric, 3) AS mean
        FROM osm_edges;
    """), conn)
print("\nDistribution finale :")
print(stats.T)

Calcul greenery_ratio par échantillonnage (rapide)...
Colonne reset en 0s
  Batch 1/9 : 46s  total 46s
  Batch 2/9 : 41s  total 87s
  Batch 3/9 : 36s  total 123s
  Batch 4/9 : 41s  total 163s
  Batch 5/9 : 42s  total 205s
  Batch 6/9 : 40s  total 245s
  Batch 7/9 : 39s  total 284s
  Batch 8/9 : 42s  total 326s
  Batch 9/9 : 23s  total 349s

✅ Calcul terminé en 349s total

Distribution finale :
                   0
ratio_0   599142.000
r_0_20      9879.000
r_20_50    10303.000
r_50_80    21063.000
r_80_100  191562.000
mean           0.255


In [6]:
with engine.connect() as conn:
    dist = pd.read_sql(text("""
        SELECT
            COUNT(*) FILTER (WHERE greenery_ratio = 0) AS ratio_0,
            COUNT(*) FILTER (WHERE greenery_ratio > 0 AND greenery_ratio <= 0.2) AS r_0_20,
            COUNT(*) FILTER (WHERE greenery_ratio > 0.2 AND greenery_ratio <= 0.5) AS r_20_50,
            COUNT(*) FILTER (WHERE greenery_ratio > 0.5 AND greenery_ratio <= 0.8) AS r_50_80,
            COUNT(*) FILTER (WHERE greenery_ratio > 0.8) AS r_80_100
        FROM osm_edges;
    """), conn)
print("Distribution APRES fix :")
print(dist.T)

with engine.connect() as conn:
    by_hw2 = pd.read_sql(text("""
        SELECT highway, ROUND(AVG(greenery_ratio)::numeric, 2) AS mean_green
        FROM osm_edges WHERE greenery_ratio IS NOT NULL
          AND highway IN ('primary','secondary','tertiary','residential',
                          'unclassified','cycleway','living_street')
        GROUP BY highway ORDER BY mean_green DESC;
    """), conn)
print("\nMoyenne par type (APRES fix) :")
print(by_hw2.to_string(index=False))

Distribution APRES fix :
               0
ratio_0   599142
r_0_20      9879
r_20_50    10303
r_50_80    21063
r_80_100  191562

Moyenne par type (APRES fix) :
      highway  mean_green
 unclassified        0.50
     tertiary        0.36
      primary        0.35
    secondary        0.32
  residential        0.20
     cycleway        0.15
living_street        0.14


**Résultat attendu** :
- `ratio_0` explose (arêtes pas du tout près de forêt/champs)
- `r_80_100` s'effondre (vraies arêtes en pleine nature)
- `residential` et `primary` tombent < 0.15 en zones urbaines
- `unclassified` et `tertiary` campagne restent 0.5-0.8

## 3. Exclusions surface 25mm

On bannit :
- surface = gravel, unpaved, dirt, ground, grass, sand, etc.
- highway = track sans asphalte explicite
- highway = path sans asphalte explicite

In [7]:
print("Application exclusions 25mm...")
with engine.begin() as conn:
    n1 = conn.execute(text("""
        UPDATE osm_edges SET is_routable = FALSE
        WHERE surface IN ('unpaved','ground','dirt','earth','mud',
                          'grass','sand','gravel','fine_gravel',
                          'pebblestone','woodchips');
    """)).rowcount
    n2 = conn.execute(text("""
        UPDATE osm_edges SET is_routable = FALSE
        WHERE highway = 'track'
          AND (surface IS NULL 
               OR surface NOT IN ('asphalt','paved','concrete','paving_stones'));
    """)).rowcount
    n3 = conn.execute(text("""
        UPDATE osm_edges SET is_routable = FALSE
        WHERE highway = 'path'
          AND (surface IS NULL 
               OR surface NOT IN ('asphalt','paved','concrete'));
    """)).rowcount

print(f"  Surfaces inroulables : {n1:,}")
print(f"  Track non-asphalté   : {n2:,}")
print(f"  Path non-asphalté    : {n3:,}")

with engine.connect() as conn:
    total = conn.execute(text("SELECT COUNT(*) FROM osm_edges")).scalar()
    routable = conn.execute(text("SELECT COUNT(*) FROM osm_edges WHERE is_routable")).scalar()
print(f"\nArêtes routables : {routable:,} / {total:,} ({100*routable/total:.1f}%)")

Application exclusions 25mm...
  Surfaces inroulables : 28,688
  Track non-asphalté   : 47,908
  Path non-asphalté    : 48,731

Arêtes routables : 407,474 / 831,949 (49.0%)


## 4. Recalcul cost_factor_club avec nouveau greenery

Bonus greenery renforcé à 0.25 (vs 0.15 avant) puisque maintenant discriminant.

In [8]:
print("Recalcul cost_factor_club...")
with engine.begin() as conn:
    conn.execute(text("""
        WITH base AS (
            SELECT e.edge_id,
                CASE
                    WHEN e.highway IN ('tertiary','unclassified','tertiary_link') THEN 1.0
                    WHEN e.highway = 'secondary' AND (e.maxspeed IS NULL OR e.maxspeed < 70) THEN 1.3
                    WHEN e.highway = 'secondary' AND e.maxspeed BETWEEN 70 AND 89 THEN 2.5
                    WHEN e.highway = 'secondary_link' THEN 1.5
                    WHEN e.highway = 'primary' AND e.maxspeed < 70 THEN 2.0
                    WHEN e.highway = 'primary' AND e.maxspeed BETWEEN 70 AND 89 THEN 5.0
                    WHEN e.highway = 'primary_link' THEN 3.0
                    WHEN e.highway = 'residential' THEN 1.2
                    WHEN e.highway = 'living_street' THEN 1.5
                    WHEN e.highway = 'cycleway' THEN 1.0
                    WHEN e.highway = 'service' THEN 2.5
                    WHEN e.highway = 'path' AND e.bicycle = 'designated' THEN 2.0
                    WHEN e.highway = 'track' THEN 2.5
                    WHEN e.highway = 'busway' THEN 1.5
                    ELSE 2.0
                END AS hierarchy_f,
                CASE
                    WHEN e.surface IN ('asphalt','paved','concrete') THEN 1.0
                    WHEN e.surface = 'paving_stones' THEN 1.4
                    WHEN e.surface = 'sett' THEN 1.8
                    WHEN e.surface IN ('cobblestone','unhewn_cobblestone') THEN 3.0
                    WHEN e.surface IS NULL THEN
                        CASE WHEN e.highway IN ('primary','secondary','tertiary','unclassified',
                                                'residential','living_street','cycleway',
                                                'primary_link','secondary_link','tertiary_link')
                             THEN 1.0 ELSE 1.5 END
                    ELSE 1.5
                END AS surface_f,
                CASE
                    WHEN e.has_bnac AND e.highway IN ('primary','primary_link') THEN 0.7
                    WHEN e.has_bnac AND e.highway IN ('secondary','secondary_link') THEN 0.85
                    WHEN e.has_bnac AND e.highway IN ('tertiary','unclassified') THEN 1.0
                    WHEN e.has_bnac AND e.highway = 'residential' THEN 1.05
                    WHEN e.has_bnac AND e.highway IN ('path','footway') THEN 1.2
                    ELSE 1.0
                END AS bnac_f,
                CASE
                    WHEN e.highway IN ('residential','living_street') THEN
                        CASE
                            WHEN e.node_density IS NULL THEN 1.0
                            WHEN e.node_density < 50 THEN 0.9
                            WHEN e.node_density < 150 THEN 1.0
                            WHEN e.node_density < 300 THEN 1.3
                            ELSE 1.6
                        END
                    ELSE 1.0
                END AS residential_f,
                CASE
                    WHEN e.greenery_ratio IS NULL THEN 1.0
                    ELSE 1.0 - 0.25 * COALESCE(e.greenery_ratio, 0)
                END AS greenery_f
            FROM osm_edges e
            WHERE e.is_routable = TRUE
        )
        UPDATE edge_scores es
        SET cost_factor_club = GREATEST(1.0,
            b.hierarchy_f * b.surface_f * b.bnac_f * b.residential_f * b.greenery_f),
            score_osm_club = 1.0 / (1.0 + (b.hierarchy_f * b.surface_f * b.bnac_f * b.residential_f * b.greenery_f - 1.0))
        FROM base b
        WHERE es.edge_id = b.edge_id;
    """))
print("cost_factor_club OK")

Recalcul cost_factor_club...
cost_factor_club OK


In [9]:
print("Recalcul cost_factor_solo...")
with engine.begin() as conn:
    conn.execute(text("""
        WITH base AS (
            SELECT e.edge_id,
                CASE
                    WHEN e.highway IN ('tertiary','unclassified','tertiary_link') THEN 1.0
                    WHEN e.highway = 'secondary' AND (e.maxspeed IS NULL OR e.maxspeed < 70) THEN 1.2
                    WHEN e.highway = 'secondary' AND e.maxspeed BETWEEN 70 AND 89 THEN 2.0
                    WHEN e.highway = 'secondary_link' THEN 1.3
                    WHEN e.highway = 'primary' AND e.maxspeed < 70 THEN 1.8
                    WHEN e.highway = 'primary' AND e.maxspeed BETWEEN 70 AND 89 THEN 4.0
                    WHEN e.highway = 'primary_link' THEN 2.5
                    WHEN e.highway = 'residential' THEN 1.1
                    WHEN e.highway = 'living_street' THEN 1.2
                    WHEN e.highway = 'cycleway' THEN 0.85
                    WHEN e.highway = 'service' THEN 2.0
                    WHEN e.highway = 'path' AND e.bicycle = 'designated' THEN 1.5
                    WHEN e.highway = 'track' THEN 2.0
                    WHEN e.highway = 'busway' THEN 1.3
                    ELSE 1.8
                END AS hierarchy_f,
                CASE
                    WHEN e.surface IN ('asphalt','paved','concrete') THEN 1.0
                    WHEN e.surface = 'paving_stones' THEN 1.2
                    WHEN e.surface = 'sett' THEN 1.5
                    WHEN e.surface IN ('cobblestone','unhewn_cobblestone') THEN 2.5
                    WHEN e.surface IS NULL THEN
                        CASE WHEN e.highway IN ('primary','secondary','tertiary','unclassified',
                                                'residential','living_street','cycleway',
                                                'primary_link','secondary_link','tertiary_link')
                             THEN 1.0 ELSE 1.3 END
                    ELSE 1.3
                END AS surface_f,
                CASE
                    WHEN e.has_bnac AND e.highway IN ('primary','primary_link') THEN 0.6
                    WHEN e.has_bnac AND e.highway IN ('secondary','secondary_link') THEN 0.8
                    WHEN e.has_bnac AND e.highway IN ('tertiary','unclassified') THEN 0.95
                    WHEN e.has_bnac AND e.highway = 'residential' THEN 0.95
                    WHEN e.has_bnac AND e.highway IN ('path','footway') THEN 1.0
                    ELSE 1.0
                END AS bnac_f,
                CASE
                    WHEN e.highway IN ('residential','living_street') THEN
                        CASE
                            WHEN e.node_density IS NULL THEN 1.0
                            WHEN e.node_density < 50 THEN 0.9
                            WHEN e.node_density < 150 THEN 1.0
                            WHEN e.node_density < 300 THEN 1.15
                            ELSE 1.3
                        END
                    ELSE 1.0
                END AS residential_f,
                CASE
                    WHEN e.greenery_ratio IS NULL THEN 1.0
                    ELSE 1.0 - 0.25 * COALESCE(e.greenery_ratio, 0)
                END AS greenery_f
            FROM osm_edges e
            WHERE e.is_routable = TRUE
        )
        UPDATE edge_scores es
        SET cost_factor_solo = GREATEST(1.0,
            b.hierarchy_f * b.surface_f * b.bnac_f * b.residential_f * b.greenery_f),
            score_osm_solo = 1.0 / (1.0 + (b.hierarchy_f * b.surface_f * b.bnac_f * b.residential_f * b.greenery_f - 1.0))
        FROM base b
        WHERE es.edge_id = b.edge_id;
    """))

# Score final v1
W_CLUB, W_OSM = 0.55, 0.45
with engine.begin() as conn:
    conn.execute(text(f"""
        UPDATE edge_scores
        SET score_final_club = {W_CLUB} * score_club + {W_OSM} * score_osm_club,
            score_final_solo = {W_CLUB} * score_club + {W_OSM} * score_osm_solo;
    """))
print("cost_factor_solo + score_final OK")

Recalcul cost_factor_solo...
cost_factor_solo + score_final OK


## 5. Recalcul cost_factor_v2 (si Notebook 5 a tourné)

In [ ]:
BETA = 0.3

with engine.connect() as conn:
    has_gen = conn.execute(text("""
        SELECT COUNT(*) FROM edge_scores WHERE score_club_gen > 0;
    """)).scalar()

if has_gen > 0:
    print(f"score_club_gen présent ({has_gen:,}), mise à jour cost_factor_v2...")
    with engine.begin() as conn:
        conn.execute(text(f"""
            UPDATE edge_scores
            SET cost_factor_v2_club = GREATEST(1.0,
                    cost_factor_club * (1 - {BETA} * COALESCE(score_club_gen, 0))),
                cost_factor_v2_solo = GREATEST(1.0,
                    cost_factor_solo * (1 - {BETA} * COALESCE(score_club_gen, 0)));
        """))
    print("cost_factor_v2 OK")
else:
    print("score_club_gen pas encore calculé, skip v2")

## 6. Bilan

In [ ]:
with engine.connect() as conn:
    summary = pd.read_sql(text("""
        SELECT
            (SELECT COUNT(*) FROM osm_edges) AS total,
            (SELECT COUNT(*) FROM osm_edges WHERE is_routable) AS routable,
            (SELECT COUNT(*) FROM osm_edges WHERE NOT is_routable) AS excluded,
            ROUND((SELECT AVG(greenery_ratio)::numeric FROM osm_edges), 3) AS greenery_mean,
            ROUND((SELECT AVG(cost_factor_club)::numeric FROM edge_scores 
                   WHERE cost_factor_club > 0), 2) AS cost_club_mean,
            ROUND((SELECT AVG(cost_factor_solo)::numeric FROM edge_scores 
                   WHERE cost_factor_solo > 0), 2) AS cost_solo_mean;
    """), conn)
print("Bilan après corrections :")
print(summary.T)

## Impact attendu sur le routing

1. **Plus de tentatives sur chemins de terre** (is_routable strict)
2. **Bonus campagne renforcé** : vraie différence entre "en forêt" et "en ville"
3. **Moins de bruit urbain** : rues de Paris ne sont plus artificiellement "vertes"

Relance le Notebook 4 pour voir les nouveaux itinéraires.

In [2]:
import time
from sqlalchemy import text

print("Détection des vrais croisements de nationales (version optimisée)...")
t0 = time.time()

with engine.begin() as conn:
    # Étape 1 : table des nœuds routables (rapide)
    print("  Étape 1/4 : safe_nodes...")
    conn.execute(text("""
        DROP TABLE IF EXISTS safe_nodes;
        CREATE TABLE safe_nodes AS
        SELECT DISTINCT n.geom FROM (
            SELECT ST_StartPoint(geom) AS geom FROM osm_edges WHERE is_routable
            UNION
            SELECT ST_EndPoint(geom) FROM osm_edges WHERE is_routable
        ) n;
        CREATE INDEX idx_safe_nodes ON safe_nodes USING GIST (geom);
        ANALYZE safe_nodes;
    """))

    # Étape 2 : table des EXTRÉMITÉS des arêtes exclues dangereuses (indexée)
    # C'est LA clé de l'optimisation : on pré-calcule les points
    print("  Étape 2/4 : excluded_endpoints...")
    conn.execute(text("""
        DROP TABLE IF EXISTS excluded_endpoints;
        CREATE TABLE excluded_endpoints AS
        SELECT edge_id, ST_StartPoint(geom) AS pt FROM osm_edges
        WHERE NOT is_routable
          AND highway IN ('primary','secondary','trunk','motorway',
                          'primary_link','secondary_link','trunk_link','motorway_link')
        UNION ALL
        SELECT edge_id, ST_EndPoint(geom) FROM osm_edges
        WHERE NOT is_routable
          AND highway IN ('primary','secondary','trunk','motorway',
                          'primary_link','secondary_link','trunk_link','motorway_link');
        
        CREATE INDEX idx_excl_ep_geom ON excluded_endpoints USING GIST (pt);
        CREATE INDEX idx_excl_ep_eid ON excluded_endpoints (edge_id);
        ANALYZE excluded_endpoints;
    """))

    # Étape 3 : candidats (arêtes courtes exclues primary/secondary)
    print("  Étape 3/4 : candidates...")
    conn.execute(text("""
        DROP TABLE IF EXISTS candidates;
        CREATE TABLE candidates AS
        SELECT e.edge_id,
               ST_StartPoint(e.geom) AS p1,
               ST_EndPoint(e.geom) AS p2
        FROM osm_edges e
        WHERE NOT e.is_routable
          AND e.highway IN ('primary', 'secondary')
          AND e.length_m < 80;
        CREATE INDEX idx_cand_p1 ON candidates USING GIST (p1);
        CREATE INDEX idx_cand_p2 ON candidates USING GIST (p2);
        ANALYZE candidates;
    """))

    # Étape 4 : filtre final
    # - p1 et p2 sont dans safe_nodes (extrémités connectées au réseau routable)
    # - Pas d'autre arête exclue adjacente à p1 ou p2
    print("  Étape 4/4 : crossings_ok...")
    conn.execute(text("""
        DROP TABLE IF EXISTS crossings_ok;
        CREATE TABLE crossings_ok AS
        SELECT c.edge_id
        FROM candidates c
        WHERE EXISTS (SELECT 1 FROM safe_nodes s WHERE ST_DWithin(s.geom, c.p1, 0.00001))
          AND EXISTS (SELECT 1 FROM safe_nodes s WHERE ST_DWithin(s.geom, c.p2, 0.00001))
          AND NOT EXISTS (
              SELECT 1 FROM excluded_endpoints ep
              WHERE ep.edge_id != c.edge_id
                AND (ST_DWithin(ep.pt, c.p1, 0.00001)
                     OR ST_DWithin(ep.pt, c.p2, 0.00001))
          );
        CREATE INDEX idx_crossings_ok ON crossings_ok (edge_id);
    """))

    # Cleanup tables intermédiaires
    conn.execute(text("DROP TABLE candidates; DROP TABLE excluded_endpoints;"))

with engine.connect() as conn:
    n_candidates_raw = conn.execute(text("""
        SELECT COUNT(*) FROM osm_edges
        WHERE NOT is_routable AND highway IN ('primary','secondary') AND length_m < 80;
    """)).scalar()
    n_crossings = conn.execute(text("SELECT COUNT(*) FROM crossings_ok")).scalar()

print(f"\n  {n_candidates_raw:,} arêtes courtes exclues au total")
print(f"  {n_crossings:,} vrais croisements identifiés (passent les 3 filtres)")
print(f"  Terminé en {time.time()-t0:.0f}s")

Détection des vrais croisements de nationales (version optimisée)...
  Étape 1/4 : safe_nodes...
  Étape 2/4 : excluded_endpoints...
  Étape 3/4 : candidates...
  Étape 4/4 : crossings_ok...

  13,985 arêtes courtes exclues au total
  13 vrais croisements identifiés (passent les 3 filtres)
  Terminé en 11s


In [3]:
print("Application de la re-autorisation...")
with engine.begin() as conn:
    # Re-autoriser
    n_ra = conn.execute(text("""
        UPDATE osm_edges
        SET is_routable = TRUE
        WHERE edge_id IN (SELECT edge_id FROM crossings_ok);
    """)).rowcount
    
    # Écraser les cost_factor par une valeur forfaitaire dissuasive
    conn.execute(text("""
        UPDATE edge_scores es
        SET cost_factor_club = 20.0,
            cost_factor_solo = 20.0,
            score_osm_club = 0.05,
            score_osm_solo = 0.05,
            score_final_club = 0.55 * score_club + 0.45 * 0.05,
            score_final_solo = 0.55 * score_club + 0.45 * 0.05
        FROM crossings_ok c
        WHERE es.edge_id = c.edge_id;
    """))
    
    # Si Notebook 5 a tourné, mettre à jour v2 aussi
    conn.execute(text("""
        UPDATE edge_scores
        SET cost_factor_v2_club = 20.0,
            cost_factor_v2_solo = 20.0
        WHERE edge_id IN (SELECT edge_id FROM crossings_ok)
          AND cost_factor_v2_club IS NOT NULL;
    """))
    
    # Nettoyage
    conn.execute(text("DROP TABLE IF EXISTS crossings_ok; DROP TABLE IF EXISTS safe_nodes;"))

print(f"  {n_ra:,} arêtes re-autorisées avec cost_factor=20")

Application de la re-autorisation...
  13 arêtes re-autorisées avec cost_factor=20


In [4]:
import folium
from shapely import wkt

with engine.connect() as conn:
    reauth = pd.read_sql(text("""
        SELECT e.edge_id, e.highway, e.length_m, e.maxspeed,
               ST_AsText(e.geom) AS wkt
        FROM osm_edges e
        JOIN edge_scores es ON es.edge_id = e.edge_id
        WHERE es.cost_factor_club = 20.0
        LIMIT 300;
    """), conn)

print(f"{len(reauth)} arêtes re-autorisées échantillonnées")

m = folium.Map(location=[48.75, 2.3], zoom_start=10, tiles="cartodbpositron")
for _, r in reauth.iterrows():
    g = wkt.loads(r["wkt"])
    coords = [(y, x) for x, y in g.coords]
    folium.PolyLine(coords, color="red", weight=4, opacity=0.8,
        tooltip=f"{r['highway']} | {r['length_m']:.0f}m | maxspeed={r['maxspeed']}"
    ).add_to(m)
m

20 arêtes re-autorisées échantillonnées


In [5]:
with engine.connect() as conn:
    diag = pd.read_sql(text("""
        SELECT
            -- Total arêtes exclues primary/secondary
            (SELECT COUNT(*) FROM osm_edges WHERE NOT is_routable 
                AND highway IN ('primary','secondary')) AS total_excluded,
            
            -- Combien font < 80m
            (SELECT COUNT(*) FROM osm_edges WHERE NOT is_routable 
                AND highway IN ('primary','secondary') AND length_m < 80) AS under_80m,
            
            -- Combien font < 150m
            (SELECT COUNT(*) FROM osm_edges WHERE NOT is_routable 
                AND highway IN ('primary','secondary') AND length_m < 150) AS under_150m,
            
            -- Combien font < 300m
            (SELECT COUNT(*) FROM osm_edges WHERE NOT is_routable 
                AND highway IN ('primary','secondary') AND length_m < 300) AS under_300m,
            
            -- Distribution de longueur des exclues
            (SELECT ROUND(AVG(length_m)::numeric, 0) FROM osm_edges WHERE NOT is_routable 
                AND highway IN ('primary','secondary')) AS mean_length,
            (SELECT MIN(length_m) FROM osm_edges WHERE NOT is_routable 
                AND highway IN ('primary','secondary')) AS min_length,
            (SELECT MAX(length_m) FROM osm_edges WHERE NOT is_routable 
                AND highway IN ('primary','secondary')) AS max_length;
    """), conn)
print(diag.T)

                           0
total_excluded  24210.000000
under_80m       13972.000000
under_150m      18248.000000
under_300m      21577.000000
mean_length       147.000000
min_length          0.532701
max_length       7861.991984


# Strava integration

In [2]:
# Réintégration Strava avec poids modeste
W_CLUB   = 0.50    # avant 0.55
W_OSM    = 0.40    # avant 0.45
W_STRAVA = 0.10    # avant 0.00

with engine.begin() as conn:
    conn.execute(text(f"""
        UPDATE edge_scores
        SET score_final_club = {W_CLUB} * score_club 
                              + {W_OSM} * score_osm_club
                              + {W_STRAVA} * COALESCE(score_strava, 0),
            score_final_solo = {W_CLUB} * score_club 
                              + {W_OSM} * score_osm_solo
                              + {W_STRAVA} * COALESCE(score_strava, 0);
    """))

with engine.connect() as conn:
    s = pd.read_sql(text("""
        SELECT
            ROUND(AVG(score_final_club)::numeric, 3) AS club_mean,
            ROUND(AVG(score_final_solo)::numeric, 3) AS solo_mean,
            COUNT(*) FILTER (WHERE score_strava > 0) AS avec_strava
        FROM edge_scores;
    """), conn)
print(s.to_string(index=False))

 club_mean  solo_mean  avec_strava
      0.15       0.18            0


In [3]:
with engine.connect() as conn:
    diag = pd.read_sql(text("""
        SELECT
            (SELECT COUNT(*) FROM strava_segments) AS segments_en_db,
            (SELECT COUNT(*) FROM edge_scores WHERE score_strava > 0) AS edges_avec_score,
            (SELECT COUNT(*) FROM edge_scores WHERE score_strava IS NULL) AS edges_score_null,
            (SELECT MAX(score_strava) FROM edge_scores) AS max_score
        FROM edge_scores LIMIT 1;
    """), conn)
print(diag.T)

                       0
segments_en_db    6130.0
edges_avec_score     0.0
edges_score_null     0.0
max_score            0.0


In [4]:
import time
from sqlalchemy import text

print("Recalcul cost_factor avec contexte urbain renforcé...")
t0 = time.time()

with engine.begin() as conn:
    # Recalcul cost_factor_club avec ajustements puissants
    conn.execute(text("""
        WITH base AS (
            SELECT e.edge_id,
                -- 1. HIERARCHIE
                CASE
                    WHEN e.highway IN ('tertiary','unclassified','tertiary_link') THEN 1.0
                    WHEN e.highway = 'secondary' AND (e.maxspeed IS NULL OR e.maxspeed < 70) THEN 1.3
                    WHEN e.highway = 'secondary' AND e.maxspeed BETWEEN 70 AND 89 THEN 2.5
                    WHEN e.highway = 'secondary_link' THEN 1.5
                    WHEN e.highway = 'primary' AND e.maxspeed < 70 THEN 2.0
                    WHEN e.highway = 'primary' AND e.maxspeed BETWEEN 70 AND 89 THEN 5.0
                    WHEN e.highway = 'primary_link' THEN 3.0
                    WHEN e.highway = 'residential' THEN 1.2
                    WHEN e.highway = 'living_street' THEN 1.5
                    WHEN e.highway = 'cycleway' THEN 1.0
                    WHEN e.highway = 'service' THEN 2.5
                    WHEN e.highway = 'path' AND e.bicycle = 'designated' THEN 2.0
                    WHEN e.highway = 'track' THEN 2.5
                    WHEN e.highway = 'busway' THEN 1.5
                    ELSE 2.0
                END AS hierarchy_f,
                
                -- 2. SURFACE
                CASE
                    WHEN e.surface IN ('asphalt','paved','concrete') THEN 1.0
                    WHEN e.surface = 'paving_stones' THEN 1.4
                    WHEN e.surface = 'sett' THEN 1.8
                    WHEN e.surface IN ('cobblestone','unhewn_cobblestone') THEN 3.0
                    WHEN e.surface IS NULL THEN
                        CASE WHEN e.highway IN ('primary','secondary','tertiary','unclassified',
                                                'residential','living_street','cycleway',
                                                'primary_link','secondary_link','tertiary_link')
                             THEN 1.0 ELSE 1.5 END
                    ELSE 1.5
                END AS surface_f,
                
                -- 3. BNAC contextuel
                CASE
                    WHEN e.has_bnac AND e.highway IN ('primary','primary_link') THEN 0.7
                    WHEN e.has_bnac AND e.highway IN ('secondary','secondary_link') THEN 0.85
                    WHEN e.has_bnac AND e.highway IN ('tertiary','unclassified') THEN 1.0
                    WHEN e.has_bnac AND e.highway = 'residential' THEN 1.05
                    WHEN e.has_bnac AND e.highway IN ('path','footway') THEN 1.2
                    ELSE 1.0
                END AS bnac_f,
                
                -- 4. MALUS DENSITÉ URBAINE (RENFORCÉ - s'applique à TOUTES les arêtes, pas seulement résidentielles)
                CASE
                    WHEN e.node_density IS NULL THEN 1.0
                    WHEN e.node_density < 50 THEN 0.95     -- village = bonus léger
                    WHEN e.node_density < 150 THEN 1.0     -- rural/périurbain = neutre
                    WHEN e.node_density < 300 THEN 1.25    -- urbain modéré = malus
                    WHEN e.node_density < 500 THEN 1.6     -- urbain dense = gros malus
                    ELSE 2.0                                -- centre-ville Paris = très gros malus
                END AS density_f,
                
                -- 5. CYCLEWAY URBAINE pénalisée
                CASE
                    WHEN e.highway = 'cycleway' AND e.node_density > 250 THEN 1.5
                    WHEN e.highway = 'cycleway' AND e.node_density > 150 THEN 1.2
                    ELSE 1.0
                END AS cycleway_urban_f,
                
                -- 6. RECOMPENSE GREENERY - augmenté à 0.4 (vraie campagne = -40%)
                CASE
                    WHEN e.greenery_ratio IS NULL THEN 1.0
                    ELSE 1.0 - 0.40 * COALESCE(e.greenery_ratio, 0)
                END AS greenery_f
                
            FROM osm_edges e
            WHERE e.is_routable = TRUE
        )
        UPDATE edge_scores es
        SET cost_factor_club = GREATEST(1.0,
            b.hierarchy_f * b.surface_f * b.bnac_f * b.density_f * b.cycleway_urban_f * b.greenery_f
        ),
            score_osm_club = 1.0 / (1.0 + (b.hierarchy_f * b.surface_f * b.bnac_f * b.density_f * b.cycleway_urban_f * b.greenery_f - 1.0))
        FROM base b
        WHERE es.edge_id = b.edge_id;
    """))
    
    # Même logique pour solo_casual mais plus doux sur cycleway et zones denses
    conn.execute(text("""
        WITH base AS (
            SELECT e.edge_id,
                CASE
                    WHEN e.highway IN ('tertiary','unclassified','tertiary_link') THEN 1.0
                    WHEN e.highway = 'secondary' AND (e.maxspeed IS NULL OR e.maxspeed < 70) THEN 1.2
                    WHEN e.highway = 'secondary' AND e.maxspeed BETWEEN 70 AND 89 THEN 2.0
                    WHEN e.highway = 'secondary_link' THEN 1.3
                    WHEN e.highway = 'primary' AND e.maxspeed < 70 THEN 1.8
                    WHEN e.highway = 'primary' AND e.maxspeed BETWEEN 70 AND 89 THEN 4.0
                    WHEN e.highway = 'primary_link' THEN 2.5
                    WHEN e.highway = 'residential' THEN 1.1
                    WHEN e.highway = 'living_street' THEN 1.2
                    WHEN e.highway = 'cycleway' THEN 0.85
                    WHEN e.highway = 'service' THEN 2.0
                    WHEN e.highway = 'path' AND e.bicycle = 'designated' THEN 1.5
                    WHEN e.highway = 'track' THEN 2.0
                    WHEN e.highway = 'busway' THEN 1.3
                    ELSE 1.8
                END AS hierarchy_f,
                CASE
                    WHEN e.surface IN ('asphalt','paved','concrete') THEN 1.0
                    WHEN e.surface = 'paving_stones' THEN 1.2
                    WHEN e.surface = 'sett' THEN 1.5
                    WHEN e.surface IN ('cobblestone','unhewn_cobblestone') THEN 2.5
                    WHEN e.surface IS NULL THEN
                        CASE WHEN e.highway IN ('primary','secondary','tertiary','unclassified',
                                                'residential','living_street','cycleway',
                                                'primary_link','secondary_link','tertiary_link')
                             THEN 1.0 ELSE 1.3 END
                    ELSE 1.3
                END AS surface_f,
                CASE
                    WHEN e.has_bnac AND e.highway IN ('primary','primary_link') THEN 0.6
                    WHEN e.has_bnac AND e.highway IN ('secondary','secondary_link') THEN 0.8
                    WHEN e.has_bnac AND e.highway IN ('tertiary','unclassified') THEN 0.95
                    WHEN e.has_bnac AND e.highway = 'residential' THEN 0.95
                    WHEN e.has_bnac AND e.highway IN ('path','footway') THEN 1.0
                    ELSE 1.0
                END AS bnac_f,
                -- Malus densité plus doux pour solo
                CASE
                    WHEN e.node_density IS NULL THEN 1.0
                    WHEN e.node_density < 50 THEN 0.95
                    WHEN e.node_density < 150 THEN 1.0
                    WHEN e.node_density < 300 THEN 1.10
                    WHEN e.node_density < 500 THEN 1.25
                    ELSE 1.4
                END AS density_f,
                -- Solo n'évite PAS les cycleway urbaines (au contraire)
                1.0 AS cycleway_urban_f,
                CASE
                    WHEN e.greenery_ratio IS NULL THEN 1.0
                    ELSE 1.0 - 0.40 * COALESCE(e.greenery_ratio, 0)
                END AS greenery_f
            FROM osm_edges e
            WHERE e.is_routable = TRUE
        )
        UPDATE edge_scores es
        SET cost_factor_solo = GREATEST(1.0,
            b.hierarchy_f * b.surface_f * b.bnac_f * b.density_f * b.cycleway_urban_f * b.greenery_f
        ),
            score_osm_solo = 1.0 / (1.0 + (b.hierarchy_f * b.surface_f * b.bnac_f * b.density_f * b.cycleway_urban_f * b.greenery_f - 1.0))
        FROM base b
        WHERE es.edge_id = b.edge_id;
    """))
    
    # Score final
    conn.execute(text("""
        UPDATE edge_scores
        SET score_final_club = 0.55 * score_club + 0.45 * score_osm_club,
            score_final_solo = 0.55 * score_club + 0.45 * score_osm_solo;
    """))

print(f"Terminé en {time.time()-t0:.0f}s")

# Diagnostic
with engine.connect() as conn:
    stats = pd.read_sql(text("""
        SELECT 
            ROUND(AVG(cost_factor_club)::numeric, 2) AS cf_club_mean,
            COUNT(*) FILTER (WHERE cost_factor_club < 1.05) AS n_perfect,
            COUNT(*) FILTER (WHERE cost_factor_club > 1.5) AS n_above_15,
            COUNT(*) FILTER (WHERE cost_factor_club > 2.5) AS n_above_25
        FROM edge_scores es JOIN osm_edges e ON e.edge_id = es.edge_id
        WHERE e.is_routable;
    """), conn)
print(stats.T)

# Distribution par highway
with engine.connect() as conn:
    by_hw = pd.read_sql(text("""
        SELECT e.highway, 
               COUNT(*) AS n,
               ROUND(AVG(es.cost_factor_club)::numeric, 2) AS cf_mean,
               COUNT(*) FILTER (WHERE es.cost_factor_club < 1.05) * 100 / COUNT(*) AS pct_perfect
        FROM osm_edges e JOIN edge_scores es ON es.edge_id = e.edge_id
        WHERE e.is_routable
          AND e.highway IN ('tertiary','secondary','primary','residential','cycleway','unclassified')
        GROUP BY e.highway ORDER BY cf_mean;
    """), conn)
print("\nCost factor par highway :")
print(by_hw.to_string(index=False))

Recalcul cost_factor avec contexte urbain renforcé...
Terminé en 107s
                      0
cf_club_mean       2.66
n_perfect      68936.00
n_above_15    262432.00
n_above_25    156493.00

Cost factor par highway :
     highway      n  cf_mean  pct_perfect
unclassified  26657     1.20           61
    tertiary  33114     1.33           45
   secondary  31384     1.57           29
 residential 135768     1.64           18
    cycleway  15887     2.43           13
     primary     13     2.59            0
